<a href="https://colab.research.google.com/github/syrashid/rl-lunar-lander/blob/main/reinforce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Update apt, install dependencies, install**

In [ ]:
!apt-get -y update
!apt-get -y install swig cmake ffmpeg
!pip -q install "gymnasium[box2d]" imageio imageio-ffmpeg

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.8 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,887 kB]
Get:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [62.6 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:14 http

**Set up Lunar Lander env**

In [ ]:
import os
import gymnasium as gym

# LunarLander is usually v3, but this fallback keeps you unblocked.
def make_lander_env():
    for env_id in ["LunarLander-v3", "LunarLander-v2"]:
        try:
            return gym.make(env_id, continuous=False), env_id
            break;
        except Exception:
            pass
    raise RuntimeError("Could not create LunarLander env (tried v3/v2).")

env, ENV_ID = make_lander_env()


print("Using env:", ENV_ID)

env.observation_space

env.action_space

env.observation_space.shape

env.action_space.n

print("Observation Space:", env.observation_space)
print("Action Space:", env.action_space)
print("Observation Space Shape:", env.observation_space.shape)
print("Action Space n:", env.action_space.n)

Using env: LunarLander-v3
Observation Space: Box([ -2.5        -2.5       -10.        -10.         -6.2831855 -10.
  -0.         -0.       ], [ 2.5        2.5       10.        10.         6.2831855 10.
  1.         1.       ], (8,), float32)
Action Space: Discrete(4)
Observation Space Shape: (8,)
Action Space Options: 4


**Policy NN**

In [ ]:
import torch
import torch.nn as nn

class PolicyNet(nn.Module):
    def __init__(self, obs_dim: int = 8, n_actions: int = 4, hidden_size: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, n_actions),  # logits (raw scores), NOT softmax
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Returns action logits.
        - If x is shape [obs_dim], returns shape [n_actions]
        - If x is shape [batch, obs_dim], returns shape [batch, n_actions]
        """
        if x.dim() == 1:
            x = x.unsqueeze(0)          # [1, obs_dim]
            logits = self.net(x)        # [1, n_actions]
            return logits.squeeze(0)    # [n_actions]
        return self.net(x)              # [batch, n_actions]


**Action Selection Helper**

In [ ]:
import numpy as np
from torch.distributions import Categorical

def select_action(policy: torch.nn.Module, obs: np.ndarray, device: torch.device | str = "cpu"):
    """
    Given a single observation (shape: [8]), returns:
      - action: Python int in {0,1,2,3}
      - log_prob: torch scalar tensor (keep as tensor for REINFORCE update)
      - entropy: torch scalar tensor (optional, useful later for exploration bonus)
    """
    # 1) Convert obs -> tensor
    obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device)

    # 2) Forward pass -> logits (shape [4])
    logits = policy(obs_t)

    # 3) Distribution from logits (softmax handled internally)
    dist = Categorical(logits=logits)

    # 4) Sample action + log_prob
    action_t = dist.sample()              # tensor scalar (0..3)
    log_prob = dist.log_prob(action_t)    # tensor scalar
    entropy = dist.entropy()              # tensor scalar

    # 5) Return action as int (env.step expects int)
    action = int(action_t.item())
    return action, log_prob, entropy


**Run episode**

In [ ]:
from typing import List, Tuple
import numpy as np
import torch

def run_episode(env, policy: torch.nn.Module, device: torch.device | str = "cpu"
               ) -> Tuple[List[torch.Tensor], List[float], List[torch.Tensor]]:
    """
    Runs ONE full episode using the current policy.

    Returns:
      - log_probs: list of torch scalar tensors (one per step)
      - rewards:   list of floats (one per step)
      - entropies: list of torch scalar tensors (one per step)  [optional but handy later]
    """
    obs, info = env.reset()
    done = False

    log_probs: List[torch.Tensor] = []
    rewards: List[float] = []
    entropies: List[torch.Tensor] = []

    while not done:
        action, log_prob, entropy = select_action(policy, obs, device=device)

        next_obs, reward, terminated, truncated, info = env.step(action)
        done = bool(terminated or truncated)

        log_probs.append(log_prob)
        rewards.append(float(reward))
        entropies.append(entropy)

        obs = next_obs

    return log_probs, rewards, entropies


**Training**

**Train REINFORCE Policy**

**Run episodes**

In [ ]:
def run_episodes(env, policy, n_episodes=3, seed=42, max_steps=2000):


returns = run_episodes(env, n_episodes=3)
env.close()

print("Returns:", returns)

Episode 0: return=-292.7, steps=116, done=True, truncated=False


/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"


Episode 1: return=-391.6, steps=80, done=True, truncated=False
Episode 2: return=-418.6, steps=93, done=True, truncated=False
Returns: [-292.70687649919034, -391.5825585693524, -418.5574146835581]


**List recordings**

In [ ]:
import glob
videos = sorted(glob.glob(f"{VIDEO_DIR}/*.mp4"))
videos

['/content/lunar_lander/videos/LunarLander-v3_random-episode-0.mp4',
 '/content/lunar_lander/videos/LunarLander-v3_random-episode-1.mp4',
 '/content/lunar_lander/videos/LunarLander-v3_random-episode-2.mp4']

**View video display**

In [ ]:
from IPython.display import Video, display

latest = videos[-1]
display(Video(latest, embed=True))
print(f"Displaying: {latest}")

Displaying: /content/lunar_lander/videos/LunarLander-v3_random-episode-2.mp4
